In [1]:
import dolfinx
from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

from Training_utils import *
from FEniCSx_PyTorch_interface import Data_to_solver, fem_solver, self_supervised_train

from dolfinx.io import XDMFFile
from mpi4py import MPI
from SPDE_problems import int_to_prblm

tset_dir = "training_set"
tset = graph_dataset(f"data/{tset_dir}/input_values")

class batched_loss_fn():
    def __init__(self, set):
        self.fsl = {}
        for G in set:
            num = G.mesh_id[0]
            with XDMFFile(MPI.COMM_WORLD, f"data/training_set/mesh_files/mesh_{G.mesh_id[0]}.xdmf", "r") as xdmf:
                mesh = xdmf.read_mesh(name="mesh")


            fs = int_to_prblm(idx=G.prblm_id, mesh=mesh)
            self.fsl[int(G.mesh_id)] = fem_solver(fs)

    def __call__(self, ptr, idx, y):
        loss_vals = [self.fsl[int(idx[i])](y[ptr[i]:ptr[i+1]] ) for i in range(len(ptr)-1)]
        return torch.stack(loss_vals).sum()
    
loss_fn = batched_loss_fn(tset)


batch_size = 15
loader = train_loader(batch_size=batch_size, set=tset)
model=AbsRestriction(GATv2)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.8, patience=50)
curr_loss = 10


In [4]:

batch_size = 1
loader = train_loader(batch_size=batch_size, set=tset)

In [ ]:

for i in range(1000):
    loss = self_supervised_train(model=model, loader=loader,loss_fn=loss_fn, optimizer=optimizer, device='cpu')
    if curr_loss > loss:    
        print(f"iteration {i}: new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/abs_GATv2.pth")
        #scheduler.step(loss)
    else:
        print(f"iteration {i}: {loss}")
        scheduler.step(loss)

        


In [2]:
import dolfinx
from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

from Training_utils import *
from FEniCSx_PyTorch_interface import Data_to_solver, fem_solver, self_supervised_train

from dolfinx.io import XDMFFile
from mpi4py import MPI
from SPDE_problems import int_to_prblm

tset_dir = "training_set_wedge"
tset = graph_dataset(f"data/{tset_dir}/input_values")

class batched_loss_fn():
    def __init__(self, set):
        self.fsl = {}
        for G in set:
            num = G.mesh_id[0]
            with XDMFFile(MPI.COMM_WORLD, f"data/training_set/mesh_files/mesh_{G.mesh_id[0]}.xdmf", "r") as xdmf:
                mesh = xdmf.read_mesh(name="mesh")


            fs = int_to_prblm(idx=G.prblm_id, mesh=mesh)
            self.fsl[int(G.mesh_id)] = fem_solver(fs)

    def __call__(self, ptr, idx, y):
        loss_vals = [self.fsl[int(idx[i])](y[ptr[i]:ptr[i+1]] ) for i in range(len(ptr)-1)]
        return torch.stack(loss_vals).sum()
    
loss_fn = batched_loss_fn(tset)


batch_size = 5
loader = train_loader(batch_size=batch_size, set=tset)
model=AbsRestriction(GATv2)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.8, patience=50)
curr_loss = 10
model.load_state_dict(torch.load("data/models/abs_GATv2.pth")['model_state'])

<All keys matched successfully>

In [3]:

for i in range(500):
    loss = self_supervised_train(model=model, loader=loader,loss_fn=loss_fn, optimizer=optimizer, device='cpu')
    if curr_loss > loss:    
        print(f"iteration {i}: new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/abs_GATv2_fine.pth")
        #scheduler.step(loss)
    else:
        print(f"iteration {i}: {loss}")
        scheduler.step(loss)

iteration 0: new loss: 0.05796574553500654
iteration 1: new loss: 0.05652438117411091
iteration 2: new loss: 0.055570375824269526
iteration 3: 0.05677300276713342
iteration 4: 0.05882218924428647
iteration 5: new loss: 0.055301270007880196
iteration 6: 0.058055450460718326
iteration 7: new loss: 0.054931784891070694
iteration 8: 0.05503482925834962
iteration 9: new loss: 0.053909114463345356
iteration 10: 0.05429458753496874
iteration 11: new loss: 0.05338278244340068
iteration 12: 0.05391530334681091
iteration 13: 0.054320995296519764
iteration 14: 0.05887311033843111
iteration 15: 0.0539224354757203
iteration 16: new loss: 0.052726723292532064
iteration 17: 0.05309583359646947
iteration 18: new loss: 0.05239975960668946
iteration 19: new loss: 0.05211544801810911
iteration 20: 0.05272492847224283
iteration 21: 0.05272129936848716
iteration 22: new loss: 0.05203962946769833
iteration 23: 0.05235384079585654
iteration 24: 0.05363441218408601
iteration 25: new loss: 0.051816375551020935

In [1]:
import dolfinx
from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

from Training_utils import *
from FEniCSx_PyTorch_interface import Data_to_solver, fem_solver, self_supervised_train

from dolfinx.io import XDMFFile
from mpi4py import MPI
from SPDE_problems import int_to_prblm

tset_dir = "q_training_set"
tset = graph_dataset(f"data/{tset_dir}/input_values")

class batched_loss_fn():
    def __init__(self, set):
        self.fsl = {}
        for G in set:
            num = G.mesh_id[0]
            with XDMFFile(MPI.COMM_WORLD, f"data/training_set/mesh_files/mesh_{G.mesh_id[0]}.xdmf", "r") as xdmf:
                mesh = xdmf.read_mesh(name="mesh")


            fs = int_to_prblm(idx=G.prblm_id, mesh=mesh)
            self.fsl[int(G.mesh_id)] = fem_solver(fs)

    def __call__(self, ptr, idx, y):
        loss_vals = [self.fsl[int(idx[i])](y[ptr[i]:ptr[i+1]] ) for i in range(len(ptr)-1)]
        return torch.stack(loss_vals).sum()
    
loss_fn = batched_loss_fn(tset)


batch_size = 5
loader = train_loader(batch_size=batch_size, set=tset)
model=AbsRestriction(GATv2)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.8, patience=50)
curr_loss = 10
model.load_state_dict(torch.load("data/models/abs_GATv2_fine.pth")['model_state'])

<All keys matched successfully>

In [2]:

for i in range(100):
    loss = self_supervised_train(model=model, loader=loader,loss_fn=loss_fn, optimizer=optimizer, device='cpu')
    if curr_loss > loss:    
        print(f"iteration {i}: new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/abs_GATv2_fine_2.pth")
        #scheduler.step(loss)
    else:
        print(f"iteration {i}: {loss}")
        scheduler.step(loss)

iteration 0: new loss: 0.11584369673119
iteration 1: 0.11693299362170251
iteration 2: new loss: 0.1152507047380027
iteration 3: 0.11740309439349517
iteration 4: 0.12070537281724114
iteration 5: new loss: 0.11383090051734257
iteration 6: new loss: 0.11373683747939367
iteration 7: 0.11583793715809065
iteration 8: 0.1148073994034462
iteration 9: 0.11425306690864545
iteration 10: 0.11435204969058077
iteration 11: 0.19476342928820145
iteration 12: 0.2029664676933578
iteration 13: 0.16736913240271178
iteration 14: 0.14030183791575723
iteration 15: 0.12279835041653087
iteration 16: 0.12151492426391997
iteration 17: 0.12034069775961598
iteration 18: 0.11924431421206805
iteration 19: 0.11818100292251188
iteration 20: 0.11719243980422789
iteration 21: 0.11534294781534982
iteration 22: 0.1143105846609842
iteration 23: 0.1144988237023199
iteration 24: 0.11422278855764034
iteration 25: 0.11401818461979474
iteration 26: 0.11423192493649026
iteration 27: new loss: 0.11340043072280537
iteration 28: ne